# Predicción de precios de viviendas — Real Estate (AWS Workshop)

Notebook equivalente al flujo de Amazon SageMaker Canvas, programado en Python tras un bloqueo de permisos en la cuenta de laboratorio (`AccessDeniedException` en DataZone e IAM Identity Center). Ver detalles completos en el `README.md` del repositorio.

## 1. Instalación de librerías (solo si el entorno no las tiene ya)

In [ ]:
# !pip install pandas seaborn scikit-learn matplotlib

## 2. Carga de Datos y Gráfica de Relaciones

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Cargamos el archivo CSV oficial del taller de AWS
df = pd.read_csv('../data/housing.csv')

# 2. Mostramos en pantalla las primeras 3 filas para comprobar los datos
print("Primeras filas del archivo del taller:")
print(df.head(3))

# 3. Dibujamos el mapa de calor usando solo las columnas numéricas
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='Dark2', fmt='.2f')
plt.title('Matriz de Correlación - Dataset Oficial de AWS')
plt.show()

## 3. Entrenamiento del Modelo de Predicción

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Limpiamos las celdas vacías para evitar fallos
df_clean = df.dropna()

# 2. Separamos las columnas de datos del precio final
X = df_clean.drop(columns=['median_house_value', 'ocean_proximity'])
y = df_clean['median_house_value']

# 3. Dividimos en grupo de estudio (80%) y grupo de examen (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Ponemos al algoritmo a estudiar los datos
model = LinearRegression()
model.fit(X_train, y_train)

print("¡Modelo entrenado con éxito usando los datos del taller!")

## 4. Examen del Modelo y Resultados de Precisión

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# El modelo intenta adivinar los precios del grupo de examen
y_pred = model.predict(X_test)

# Calculamos las notas de rendimiento finales
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Error Cuadrático Medio (MSE): {mse:.4f}")
print(f"Porcentaje de acierto o Coeficiente (R2 Score): {r2:.4f}")

## 5. Análisis Gráfico de los Resultados

In [ ]:
# Creamos una pantalla con dos gráficas paralelas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Precios Reales vs Predicciones
ax1.scatter(y_test, y_pred, alpha=0.3, color='teal')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_xlabel('Precios Reales del Mercado')
ax1.set_ylabel('Precios Predictivos del Modelo')
ax1.set_title('Comparativa: Realidad vs Predicción')

# Gráfica 2: Distribución de los Errores (Residuos)
errores = y_test - y_pred
sns.histplot(errores, kde=True, ax=ax2, color='purple')
ax2.axvline(x=0, color='r', linestyle='--')
ax2.set_xlabel('Tamaño del Error')
ax2.set_ylabel('Cantidad de Casas')
ax2.set_title('Distribución de los Errores del Modelo')

plt.tight_layout()
plt.show()